# DiCo-NLI 第五组实验：mDeBERTa 上的蒸馏、R-Drop、方向版 R-Drop、SCL、MoCo

只有两个代码单元：配置单元和主体单元。在阿里云 PAI-DSW 上用配套的 `dico_a10_mdeberta.sh` 在终端里跑 `.py`；在 Kaggle / Colab 上可以直接 Run All（先在配置单元里设 `STAGE`）。

**做什么**：在第二组定下的 mDeBERTa 12 轮配方上，一次只加一种方法，和同一环境里重跑的只用交叉熵的基线（C0）按种子配对比较。所有方法的超参数事先固定，见配置单元；结论只说"在这组设置下"。

| 编号 | 方法目录 | 损失 |
|---|---|---|
| C0 | `ce` | 交叉熵（和第二组 e12 相同） |
| B2 | `kd` | 0.5 × 交叉熵 + 0.5 × T² × KL(老师 ‖ 学生)，T = 2；老师是第四组英语 3 轮 Qwen SFT 的同种子模型，只用原始的 2,190 行（保留重复） |
| C1 | `rdrop` | 同一批输入前向两次：(CE₁ + CE₂) / 2 + α × 对称 KL，α = 1 |
| C2 | `dir_rdrop` | 交叉熵 + α × 对称 KL(p(A, B), F/B 对调后的 p(B, A))，α = 1，只对金标是 E / F / B 的行；反向输入不算交叉熵 |
| C3 | `scl` | 0.7 × 交叉熵 + 0.3 × 监督对比损失（τ = 0.3），用最后一层 [CLS] 加一个两层投影（投影是我们的选择） |
| C4 | `moco_m0.99`、`moco_m0.999` | 0.7 × 交叉熵 + 0.3 × 监督对比损失（τ = 0.3），键来自动量编码器（含投影头）和队列（4,096 个）；同一样本不当自己的正例，没有正例的样本跳过对比项 |

共同的配方（和第二组 `e12` 相同）：`microsoft/mdeberta-v3-base`，`[CLS] text1 [SEP] text2 [SEP]`，最长 96 个 token，lr 2e-5，batch 32，前 10% 步数升温后线性衰减，weight decay 0.01（偏置和 LayerNorm 不加），梯度裁剪 1.0，最多 12 轮，每轮在内层验证上按原方向 WF1 选轮（同分取早的），推理用 fp32、相同输入只算一次。数据划分和第二组、第四组相同：按英语训练集的家族分 5 折（种子 2027），每折里再按家族分出 10% 作内层验证；四个赛道用同样的 pair_id。

**和第二组 notebook（`dico_diagnostics`）的区别**：

1. 混合精度：支持 bf16 的卡（A10、4090）用 bf16、不用梯度缩放；T4 仍用 fp16。精度、卡、平台、torch 版本记在 `run_meta.json` 和每次运行的 `done.json` 里。
2. 每次运行（赛道 × 方法 × 种子 × 折）是一个独立任务，有自己的 `task.json`（全部训练变量），跑完写 `done.json`；被打断的任务从头重跑（一次只要几分钟）。
3. 模型快照：第一次运行时记下 Hugging Face 上 `main` 的 commit，之后固定用它。在 PAI-DSW 上先试 ModelScope，下载的文件和这个快照逐个核对，对不上或下载失败就改用 hf-mirror。
4. fold 0 的模型同时评官方 dev（训练 2,190 行），可以和同样训练数据量的 Qwen 比较。
5. 更新步：只在优化器真正更新了参数之后，才推进学习率日程、MoCo 的动量更新和入队；fp16 被梯度缩放跳过的更新记在 `skipped_updates` 里。第二组在 fp16 跳过更新时也推进了学习率（只影响 T4 上的运行，旧结果不因此作废）。
6. 数值防护：编码器的前向在 autocast 里，归一化、相似度矩阵和各项损失都关掉 autocast 用 fp32 算。bf16 下出现非有限的损失、梯度或 logits，或者梯度元素有限但总范数溢出时，参数不会被更新，整次改用 fp32 重跑，`done.json` 的 `precision` 会写明"失败后重跑"；fp32 也失败就报错停下。NaN 不会经过 argmax 变成一个正常的标签。
7. 每个任务的每次尝试（开始、完成、失败、被打断、发现上次没跑完）都记在 `attempts.jsonl`，`done.json` 里有这次的 `attempt_id`、之前尝试过几次、环境身份（平台、卡、精度、torch 等）和代码哈希。不同代码或不同精度跑出来的折，不拼成同一份折外预测。
8. 蒸馏的老师：读入时逐项核对身份（导出版本、来源运行和代码、每个种子的 LoRA 权重在三个文件里一致并等于写死的值、`teacher.jsonl` 的 SHA256），拒绝重复的输入和无效的概率。

**三个阶段**（`STAGE`，PAI-DSW 上由 sh 脚本传入）：

- `smoke`：用真实的 torch 核对各个损失（和逐条循环的参考实现比、autocast 里仍是 fp32）和非有限梯度时的更新逻辑；核对老师的身份；在单独的目录里模拟一次中断再续跑；每种方法在小子集上跑 2 轮；再用真实规模的数据和 12 轮的学习率日程测每种方法每步的秒数（MoCo 先填满队列）、载入和推理的用时，打印两个阶段的时间估计。可以重复运行。约 10–15 分钟（另加下载模型）。
- `stage1`：英语 fold 0、种子 42–44：C0、B2、C1、C2、C3、C4 的两个 m；再加西班牙语、巴斯克语、跨语言的 C0（fold 0，3 个种子）。
- `stage2`：英语 5 折、种子 42–46：C0、C1、C2、C3 和选定 m 的 C4（第一阶段已完成的会跳过）。每个种子把 5 折拼成 3,042 行，再用官方脚本算一次分数。

**C4 的 m 怎么选（事先定好）**：第一阶段两个 m 各跑 3 个种子，比较选中那一轮的内层验证原方向 WF1 的三种子平均（不看 fold 0、dev 和老师），大的胜出，相同取 0.99。第二阶段开始时自动按这个规则选，并写进 `moco_m_choice.json`。

**时间**：SMOKE 会按实测打印。第二组在 T4 上每个 12 轮的英语运行约 3.1 分钟（每轮 14.7 秒）。

In [ ]:
import os
import sys
from pathlib import Path

if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    raise RuntimeError("CUDA 已初始化。请重启会话，再从头运行。")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

STAGE = os.environ.get("DICO_STAGE", "smoke")  # smoke / stage1 / stage2；notebook 里直接改这里
MODEL = "microsoft/mdeberta-v3-base"
SEEDS_STAGE1 = [42, 43, 44]
SEEDS_STAGE2 = [42, 43, 44, 45, 46]

# 共同的训练配方（和第二组 e12 相同）
MAX_EPOCHS = 12
LR = 2e-5
BATCH_SIZE = 32
EVAL_BATCH = 64
MAX_LEN = 96
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
FOLDS, FOLD_SEED, INNER_FRACTION = 5, 2027, 0.1

# 各方法的超参数（事先固定，不在结果出来后调）
KD = {"alpha": 0.5, "T": 2.0}  # alpha 是 KL 项的权重
RDROP = {"alpha": 1.0}
DIR_RDROP = {"alpha": 1.0, "labels": ["EQUIVALENCE", "FORWARD_ENTAILMENT", "BACKWARD_ENTAILMENT"]}
PROJ = {"hidden": 768, "out": 128}  # [CLS] → Linear → ReLU → Linear → L2 归一化
SCL = {"lambda": 0.3, "tau": 0.3}
MOCO = {"lambda": 0.3, "tau": 0.3, "queue": 4096, "m_candidates": [0.99, 0.999], "m_tie": 0.99}
# 老师（蒸馏第 1 步导出的概率）的身份：来源运行、来源代码、每个种子的 LoRA 权重和导出文件，全部写死，读入时逐项核对
TEACHER_VERSION = "dico_qwen_teacher-3"
TEACHER_SOURCE_FINGERPRINT = "fcb2485b743956822b75c3703943ec806bc22ba955897c87082d0e6b8608566c"  # 第四组英语 3 轮
TEACHER_SOURCE_CODE = "2863de05aa757a80170da508d950ba8fa2dc2be579ef918642b249eb45f899ad"
TEACHER_ADAPTER_SHA256 = {42: "50092805a194da00505b2d0ef90435725cbe20e2f44a9e3c9fc5463fcc1874d3",
                          43: "439e235075455d35752908ace95571ec9035ab3d49e99a0ebc3cd01167bc1720",
                          44: "ef9b469fa52fff393dd2c4022aa02da381d6de2006236f854b44ca928766f75f"}
TEACHER_JSONL_SHA256 = {42: "52f6a4e7dcaade1ecd0f6fcc27b06b9343bb3664363574e7b4bb1e8c6884efe4",
                        43: "5040d206216e6687056349572e70ccca88d27a84cdc967fe58b4de2ccd887aa2",
                        44: "e3b10815607d3e9989f08105284e8f3256dae0cb6426684d72662926ebbafaa5"}

SMOKE_ROWS = {"fit": 96, "val": 32, "test": 48, "dev": 48}
SMOKE_EPOCHS = 2
TIMING_STEPS = 12  # SMOKE 测速：每种方法用真实规模的 batch 跑多少步（前 3 步不计）

ON_KAGGLE = Path("/kaggle/working").is_dir() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
ON_COLAB = not ON_KAGGLE and Path("/var/colab/hostname").exists()
ON_PAI = not ON_KAGGLE and not ON_COLAB and Path("/mnt/workspace").is_dir()
if "DICO_WORK" in os.environ:
    WORK = os.path.abspath(os.environ["DICO_WORK"])
elif ON_PAI:
    WORK = "/mnt/workspace/dico_mdeberta"
elif ON_KAGGLE:
    WORK = "/kaggle/working"
else:
    WORK = os.path.abspath(".")
PLATFORM = os.environ.get("DICO_PLATFORM") or ("Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "阿里云 PAI-DSW" if ON_PAI else "其他 Linux")
os.makedirs(WORK, exist_ok=True)
if ON_PAI:
    os.environ.setdefault("HF_HOME", os.path.join(WORK, "hf_cache"))
if not ON_KAGGLE and not ON_COLAB and "HF_ENDPOINT" not in os.environ:
    import urllib.request
    try:
        urllib.request.urlopen("https://huggingface.co/api/models/" + MODEL, timeout=10)
    except Exception as e:
        os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
        os.environ["HF_HUB_DISABLE_XET"] = "1"
        print(f"连不上 huggingface.co（{type(e).__name__}），改用镜像 https://hf-mirror.com")
DICO_DIR = os.environ.get("DICO_DIR", os.path.join(WORK, "SemEval-2027-Task-2-DiCo-NLI"))
TEACHER_DIR = os.environ.get("DICO_TEACHER", os.path.join(WORK, "dico_qwen_teacher_results"))
OUT = os.path.join(WORK, "dico_mdeberta_smoke" if STAGE == "smoke" else "dico_mdeberta_results")
print("运行平台：", PLATFORM, "；阶段：", STAGE, "；结果目录：", OUT)

In [ ]:
import csv
import glob
import hashlib
import importlib.metadata
import importlib.util
import json
import math
import platform
import random
import shutil
import subprocess
import time
import zipfile
from collections import Counter

VERSION = "dico_mdeberta_methods-3"
LABELS = ["EQUIVALENCE", "FORWARD_ENTAILMENT", "BACKWARD_ENTAILMENT", "NEGATIVE_OTHER"]
L2I = {l: i for i, l in enumerate(LABELS)}
SWAP = [0, 2, 1, 3]  # 对调两句后：FORWARD ↔ BACKWARD，其余不变
METRICS = ["weighted_f1", "soft_cons", "hard_cons"]
REFERENCE_FIELDS = ["instance_id", "pair_id", "reverse_pair_id", "label"]  # 官方评分脚本用 csv.Sniffer 猜格式，只写这四列
RUN_STAMP = time.strftime("%Y%m%d-%H%M")
STAGE1_TRACK1 = ["ce", "kd", "rdrop", "dir_rdrop", "scl", "moco_m0.99", "moco_m0.999"]
assert STAGE in ("smoke", "stage1", "stage2"), STAGE
SMOKE = STAGE == "smoke"


# ---------- 文件、数据 ----------

def read_rows(path):
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write_rows(path, rows, fields):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()


def write_json(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)


def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def json_sha(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()


def pip_install(*pkgs):
    index = ["-i", "http://mirrors.cloud.aliyuncs.com/pypi/simple/", "--trusted-host", "mirrors.cloud.aliyuncs.com"] if ON_PAI else []
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *index, *pkgs], check=True)


def ensure_packages():
    missing = [p for p, m in (("sentencepiece", "sentencepiece"), ("protobuf", "google.protobuf")) if importlib.util.find_spec(m.split(".")[0]) is None]
    if missing:
        pip_install(*missing)


def ensure_official_repo():
    """先找已解压的官方仓库；没有就找 WORK 里官方仓库的 zip（PAI-DSW 上连 GitHub 很慢，由上传包带来），最后才克隆。"""
    if os.path.isdir(os.path.join(DICO_DIR, "final_data")):
        return
    zips = sorted(glob.glob(os.path.join(WORK, "SemEval-2027-Task-2-DiCo-NLI*.zip")))
    if zips:
        tmp = DICO_DIR + ".unzip"
        shutil.rmtree(tmp, ignore_errors=True)
        with zipfile.ZipFile(zips[-1]) as z:
            z.extractall(tmp)
        roots = [os.path.dirname(os.path.dirname(p)) for p in glob.glob(os.path.join(tmp, "**", "final_data", "train"), recursive=True)]
        if len(roots) != 1:
            raise SystemExit(f"{zips[-1]} 里找不到唯一的 final_data 目录")
        shutil.move(roots[0], DICO_DIR)
        shutil.rmtree(tmp, ignore_errors=True)
        print("已从 zip 解压官方仓库：", zips[-1])
        return
    try:
        res = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ilopezgazpio/SemEval-2027-Task-2-DiCo-NLI.git", DICO_DIR],
                             capture_output=True, text=True, timeout=600)
    except subprocess.TimeoutExpired:
        res = subprocess.CompletedProcess([], 1, "", "git clone 超过 10 分钟没完成")
    if res.returncode != 0:
        shutil.rmtree(DICO_DIR, ignore_errors=True)
        raise SystemExit(f"克隆官方仓库失败：{(res.stdout + res.stderr).strip()[-300:]}\n"
                         f"请把官方仓库的 zip（GitHub 上 Code → Download ZIP）放进 {WORK}，再重新运行。")


def data_file(track, split, kind):
    return os.path.join(DICO_DIR, "final_data", split, f"dico_nli_{split}_track{track}_{kind}.csv")


def data_hashes(track):
    return {os.path.relpath(data_file(track, s, k), DICO_DIR): sha256(data_file(track, s, k))
            for s in ("train", "dev") for k in ("participant_labeled", "reference")}


# ---------- 家族、折、划分（和第二组、第四组逐行相同） ----------

def norm(s):
    return " ".join(s.lower().split())


def build_families(rows):
    parent, first = {}, {}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for r in rows:
        p = r["pair_id"]
        parent.setdefault(p, p)
        k = tuple(sorted((norm(r["text1"]), norm(r["text2"]))))
        if k in first:
            a, b = find(p), find(first[k])
            if a != b:
                parent[max(a, b)] = min(a, b)
        else:
            first[k] = p
    return {p: find(p) for p in parent}


def assign_folds(rows, fam_of):
    size = Counter(fam_of[r["pair_id"]] for r in rows)
    fams = sorted(size)
    random.Random(FOLD_SEED).shuffle(fams)
    load, fold_of = [0] * FOLDS, {}
    for f in fams:
        k = min(range(FOLDS), key=lambda i: (load[i], i))
        fold_of[f] = k
        load[k] += size[f]
    return fold_of


def pick_families(families, fraction, seed):
    fams = sorted(set(families))
    random.Random(seed).shuffle(fams)
    return set(fams[: max(1, int(len(fams) * fraction))])


_SPLIT = {}


def split_base():
    if not _SPLIT:
        train1 = read_rows(data_file(1, "train", "participant_labeled"))
        fam_of = build_families(train1)
        _SPLIT.update(fam_of=fam_of, fold_of=assign_folds(train1, fam_of))
    return _SPLIT["fam_of"], _SPLIT["fold_of"]


def smoke_pairs(rows, n):
    """按 pair 整对抽样（反向行都在子集里，官方评分脚本才能算一致性），先保证四类各一对。"""
    by_pair = {}
    for r in rows:
        by_pair.setdefault(r["pair_id"], []).append(r)
    order = sorted(by_pair)
    random.Random(FOLD_SEED).shuffle(order)
    keep, count = [], 0
    for label in LABELS:
        p = next((p for p in order if p not in keep and any(r["label"] == label for r in by_pair[p])), None)
        if p is not None:
            keep.append(p)
            count += len(by_pair[p])
    for p in order:
        if count >= n:
            break
        if p not in keep:
            keep.append(p)
            count += len(by_pair[p])
    keep = set(keep)
    return [r for r in rows if r["pair_id"] in keep]


def parts_for(track, fold, with_dev, smoke=SMOKE):
    """fit / val / test（被预测的那一折）/ dev。内层验证的种子和第二组相同：FOLD_SEED * 100 + fold + 1。"""
    fam_of, fold_of = split_base()
    train = read_rows(data_file(track, "train", "reference"))  # reference 比 participant_labeled 多一列 reverse_pair_id，其余相同
    rest = {fam_of[r["pair_id"]] for r in train if fold_of[fam_of[r["pair_id"]]] != fold}
    inner = pick_families(rest, INNER_FRACTION, FOLD_SEED * 100 + fold + 1)
    parts = {"fit": [], "val": [], "test": []}
    for r in train:
        f = fam_of[r["pair_id"]]
        parts["test" if fold_of[f] == fold else "val" if f in inner else "fit"].append(r)
    if with_dev:
        parts["dev"] = read_rows(data_file(track, "dev", "reference"))
    if smoke:
        parts = {k: smoke_pairs(v, SMOKE_ROWS[k]) for k, v in parts.items()}
    return parts


def needed_inputs(rows, both=True):
    need, seen = [], set()
    for r in rows:
        for x in (((r["text1"], r["text2"]), (r["text2"], r["text1"])) if both else ((r["text1"], r["text2"]),)):
            if x not in seen:
                seen.add(x)
                need.append(x)
    return need


# ---------- 老师概率（B2） ----------

def load_teacher(seed, fit_rows):
    """第四组英语 3 轮 Qwen 的同种子老师在 fit 行上的四类概率（顺序 E / F / B / O，和 LABELS 相同）。
    逐项核对身份：导出版本、来源运行和代码、LoRA 权重（fingerprint、adapter.json、done.json 三处一致，并等于写死的值）、
    teacher.jsonl 的 SHA256；拒绝重复的输入、分区不对和无效的概率。按传入的 fit 行逐行映射（保留重复）。返回 (每行的概率, 文件 SHA256)。"""
    d = os.path.join(TEACHER_DIR, "track1", f"seed{seed}")
    paths = {k: os.path.join(d, k) for k in ("teacher.jsonl", "adapter.json", "done.json")}
    fp_path = os.path.join(TEACHER_DIR, "fingerprint.json")
    for path in [fp_path, *paths.values()]:
        if not os.path.exists(path):
            raise FileNotFoundError(f"找不到老师概率的文件：{path}")
    fp, adapter, done = read_json(fp_path), read_json(paths["adapter.json"]), read_json(paths["done.json"])
    want = TEACHER_ADAPTER_SHA256.get(seed)
    checks = {"导出版本": fp.get("version") == TEACHER_VERSION, "来源运行": fp.get("source_fingerprint_sha256") == TEACHER_SOURCE_FINGERPRINT,
              "来源代码": fp.get("source_code_sha256") == TEACHER_SOURCE_CODE, "不是 SMOKE": fp.get("smoke") is False,
              "fingerprint 里的权重": fp.get("adapter_sha256", {}).get(str(seed)) == want,
              "adapter.json 里的权重": adapter.get("adapter_sha256") == want, "done.json 里的权重": done.get("adapter_sha256") == want,
              "teacher.jsonl 的 SHA256": sha256(paths["teacher.jsonl"]) == TEACHER_JSONL_SHA256.get(seed)}
    bad = [k for k, ok in checks.items() if not ok]
    if bad:
        raise SystemExit(f"seed{seed} 的老师身份核对不过：{bad}。不用这份老师概率。")
    table = {}
    with open(paths["teacher.jsonl"], encoding="utf-8") as f:
        for n, line in enumerate(f, 1):
            x = json.loads(line)
            key, p = (x["a"], x["b"]), x["p"]
            if key in table or x["part"] not in ("fit", "val"):
                raise SystemExit(f"teacher.jsonl 第 {n} 行：输入重复或分区不对（{x['part']}）")
            if len(p) != 4 or not all(math.isfinite(v) and 0 <= v <= 1 for v in p) or abs(sum(p) - 1) > 1e-4:
                raise SystemExit(f"teacher.jsonl 第 {n} 行的概率无效：{p}")
            table[key] = (x["part"], p)
    probs = []
    for r in fit_rows:
        part, p = table.get((r["text1"], r["text2"]), (None, None))
        if part != "fit":
            raise SystemExit(f"老师概率里没有这一行的 fit 记录：{r['instance_id']}")
        probs.append(p)
    acc = sum(max(range(4), key=lambda i: p[i]) == L2I[r["label"]] for p, r in zip(probs, fit_rows)) / len(fit_rows)
    if len(fit_rows) >= 1000 and acc < 0.9:  # 老师见过这些行，准确率应在 .97 左右；太低说明四类的顺序或映射错了
        raise SystemExit(f"seed{seed} 的老师在 fit 行上的准确率只有 {acc:.3f}，四类的顺序或行的映射可能不对")
    return probs, TEACHER_JSONL_SHA256[seed]


# ---------- 方法定义、任务 ----------

def method_params(method):
    if method == "ce":
        return {"loss": "ce"}
    if method == "kd":
        return {"loss": "0.5*CE + 0.5*T^2*KL(teacher_T || student_T)", **KD,
                "teacher": "第四组英语 3 轮 Qwen SFT，同种子，fit 行上的四类概率", "teacher_source_fingerprint": TEACHER_SOURCE_FINGERPRINT}
    if method == "rdrop":
        return {"loss": "(CE1+CE2)/2 + alpha*symKL(p1,p2)", **RDROP}
    if method == "dir_rdrop":
        return {"loss": "CE(A,B) + alpha*symKL(p(A,B), swap(p(B,A)))；反向输入只加给金标在 labels 里的行，KL 在这些行上平均，反向输入不算 CE", **DIR_RDROP}
    if method == "scl":
        return {"loss": "(1-lambda)*CE + lambda*SupCon(batch)", **SCL, "proj": PROJ, "rep": "最后一层 [CLS]"}
    if method.startswith("moco_m"):
        m = float(method[len("moco_m"):])
        return {"loss": "(1-lambda)*CE + lambda*SupCon(q, 动量编码器的键 + 队列)", "lambda": MOCO["lambda"], "tau": MOCO["tau"],
                "queue": MOCO["queue"], "m": m, "proj": PROJ, "rep": "最后一层 [CLS]",
                "target": "动量编码器（含投影头）eval 模式，不回传梯度；只在优化器真正更新后做动量更新和入队；同一训练记录（行编号）的当前键和历史键都不当它自己的正例，文本相同的不同记录算不同样本"}
    raise ValueError(method)


def task_id(t):
    return f"track{t['track']}/{t['method']}/seed{t['seed']}/fold{t['fold']}"


def task_dir(t):
    return os.path.join(OUT, task_id(t))


def make_task(track, method, seed, fold, with_dev):
    return {"track": track, "method": method, "seed": seed, "fold": fold, "with_dev": with_dev}


def choose_m():
    """C4 的 m：第一阶段两个 m 各 3 个种子，选中那一轮的内层验证 WF1 的平均，大的胜出，相同取 MOCO['m_tie']。"""
    path = os.path.join(OUT, "moco_m_choice.json")
    if os.path.exists(path):
        return read_json(path)["m"]
    means = {}
    for m in MOCO["m_candidates"]:
        vals = []
        for s in SEEDS_STAGE1:
            d = os.path.join(OUT, f"track1/moco_m{m}/seed{s}/fold0/done.json")
            if not os.path.exists(d):
                raise SystemExit(f"第二阶段要先选 m，但第一阶段的 moco_m{m} seed{s} 还没跑完。请先跑完第一阶段。")
            vals.append(read_json(d)["val_weighted_f1"])
        means[m] = sum(vals) / len(vals)
    best = max(means.values())
    winners = [m for m in MOCO["m_candidates"] if abs(means[m] - best) < 1e-12]
    m = MOCO["m_tie"] if len(winners) > 1 else winners[0]
    write_json(path, {"m": m, "inner_val_wf1_mean": means, "rule": choose_m.__doc__})
    print("C4 的 m 按事先定的规则选为", m, "；内层验证 WF1 的三种子平均：", means)
    return m


def stage_tasks(stage):
    if stage == "smoke":
        return ([make_task(1, m, 42, 0, True) for m in STAGE1_TRACK1]
                + [make_task(2, "ce", 42, 0, True), make_task(4, "ce", 42, 0, True)])
    if stage == "stage1":
        tasks = [make_task(1, m, s, 0, True) for s in SEEDS_STAGE1 for m in STAGE1_TRACK1]
        tasks += [make_task(t, "ce", s, 0, True) for t in (2, 3, 4) for s in SEEDS_STAGE1]
        return tasks
    m = choose_m()
    methods = ["ce", "rdrop", "dir_rdrop", "scl", f"moco_m{m}"]
    return [make_task(1, meth, s, k, k == 0) for s in SEEDS_STAGE2 for k in range(FOLDS) for meth in methods]


# ---------- 指纹、运行记录、结果包 ----------

def common_fingerprint():
    return {"version_family": "dico_mdeberta_methods", "model": MODEL,
            "train": {"max_epochs": MAX_EPOCHS, "lr": LR, "batch": BATCH_SIZE, "eval_batch": EVAL_BATCH, "max_len": MAX_LEN,
                      "warmup_ratio": WARMUP_RATIO, "weight_decay": WEIGHT_DECAY, "grad_clip": GRAD_CLIP,
                      "selection": "内层验证原方向 WF1，同分取早的"},
            "split": {"folds": FOLDS, "fold_seed": FOLD_SEED, "inner_fraction": INNER_FRACTION, "inner_seed": "FOLD_SEED*100+fold+1"},
            "smoke": SMOKE}


def task_fingerprint(t, revision, extra=None):
    fp = {**common_fingerprint(), "model_revision": revision, "track": t["track"], "fold": t["fold"], "seed": t["seed"],
          "method": t["method"], "method_params": method_params(t["method"]), "with_dev": t["with_dev"],
          "data_sha256": data_hashes(t["track"]),
          "epochs": SMOKE_EPOCHS if SMOKE else MAX_EPOCHS, "smoke_rows": SMOKE_ROWS if SMOKE else None}
    fp.update(extra or {})
    return fp


def restore_previous_results():
    """第二阶段在新实例上跑时，从上传包里带来的第一阶段结果包恢复（只认设置相同、完成最多的那个）。"""
    if SMOKE or os.path.isdir(OUT):
        return
    base = os.path.basename(OUT)
    found = sorted(set(glob.glob(os.path.join(WORK, f"{base}_*.zip")) + glob.glob(f"/kaggle/input/**/{base}_*.zip", recursive=True)))
    best, best_n = None, -1
    for path in found:
        try:
            with zipfile.ZipFile(path) as z:
                old = json.loads(z.read(f"{base}/fingerprint.json"))
                if {k: v for k, v in old.items() if k != "model_revision"} != common_fingerprint():
                    print("  跳过设置不同的结果包：", path)
                    continue
                n = sum(x.endswith("done.json") for x in z.namelist())
        except (KeyError, zipfile.BadZipFile, json.JSONDecodeError):
            print("  跳过读不了的结果包：", path)
            continue
        if n > best_n:
            best, best_n = path, n
    if best:
        zipfile.ZipFile(best).extractall(WORK)
        print(f"已从结果包恢复（已完成 {best_n} 项）：", best)


def check_common_fingerprint():
    path = os.path.join(OUT, "fingerprint.json")
    fp = common_fingerprint()
    if os.path.exists(path):
        old = read_json(path)
        if {k: v for k, v in old.items() if k != "model_revision"} != fp:
            if SMOKE:
                shutil.rmtree(OUT)
            else:
                raise SystemExit(f"{path} 的设置和现在的不同。要接着跑，请把设置改回去；要重新开始，请删除或改名 {OUT}。")
    os.makedirs(OUT, exist_ok=True)
    if not os.path.exists(path):
        write_json(path, fp)
    return read_json(path)


def code_sha256():
    try:
        cells = [c for c in get_ipython().user_ns.get("In", []) if f'VERSION = "{VERSION}"' in c]  # noqa: F821
        if cells:
            return hashlib.sha256(cells[-1].encode("utf-8")).hexdigest(), "notebook cell"
    except NameError:
        pass
    path = globals().get("__file__")
    if path and os.path.isfile(path):
        return sha256(path), "file"
    return None, None


def check_code(code_hash):
    """同一个结果目录只用同一份代码；SMOKE 的目录代码变了就清空重来。"""
    path = os.path.join(OUT, "run_meta.json")
    if code_hash is None:
        if not SMOKE:
            raise SystemExit("取不到这次运行的代码哈希，正式运行先停下。请把输出发给 Claude。")
        return
    old = {r.get("code_sha256") for r in (read_json(path) if os.path.exists(path) else [])} - {None}
    if not old or old == {code_hash} or os.environ.get("DICO_ALLOW_NEW_CODE") == code_hash:
        return
    if SMOKE:
        fp = read_json(os.path.join(OUT, "fingerprint.json"))
        shutil.rmtree(OUT)
        write_json(os.path.join(OUT, "fingerprint.json"), fp)
        return
    raise SystemExit(f"这个结果目录是用另一份代码（{sorted(old)}）跑的，现在的代码哈希是 {code_hash}。不把新旧代码的输出混在一起。"
                     f"要重新开始，请删除或改名 {OUT}。")


def env_info():
    import torch
    info = {"torch": torch.__version__, "cuda": torch.version.cuda, "python": platform.python_version(),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
    for name in ("transformers", "sentencepiece", "huggingface_hub", "modelscope", "numpy"):
        try:
            info[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            info[name] = None
    return info


def record_run(info):
    path = os.path.join(OUT, "run_meta.json")
    code_hash, source = code_sha256()
    check_code(code_hash)
    runs = read_json(path) if os.path.exists(path) else []
    runs.append({"start": RUN_STAMP, "version": VERSION, "stage": STAGE, "code_sha256": code_hash, "code_source": source,
                 "platform": PLATFORM, "hf_endpoint": os.environ.get("HF_ENDPOINT", "https://huggingface.co"), **info})
    write_json(path, runs)
    print("代码哈希：", code_hash, f"（{source}）")
    return code_hash


_LAST_PACK = [0.0]


def pack(force=True):
    """结果包：整个结果目录（只有 logits、预测、分数和记录，没有模型权重），加上 sh 的日志。"""
    if not force and time.time() - _LAST_PACK[0] < 600:  # 跑的过程中最多每 10 分钟更新一次
        return None
    _LAST_PACK[0] = time.time()
    base = os.path.basename(OUT)
    zip_path = os.path.join(WORK, f"{base}_{RUN_STAMP}.zip")
    tmp = zip_path + ".tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(glob.glob(os.path.join(OUT, "**", "*"), recursive=True)):
            if os.path.isfile(path) and not path.endswith(".tmp"):
                z.write(path, os.path.relpath(path, WORK))
        log = os.environ.get("DICO_LOG")
        if log and os.path.exists(log):
            z.write(log, f"{base}/logs/{os.path.basename(log)}")
    os.replace(tmp, zip_path)
    return zip_path


# ---------- 模型下载（固定快照） ----------

WEIGHT_PATTERNS = ["*.json", "*.model", "*.safetensors", "pytorch_model.bin"]


def wanted(name):
    import fnmatch
    return "/" not in name and any(fnmatch.fnmatch(name, p) for p in WEIGHT_PATTERNS)


def file_digest(path, algo, prefix=b""):
    h = hashlib.new(algo, prefix)
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()


def fetch_model(revision):
    """返回 (本地模型目录, Hugging Face 快照的 commit, 来源)。revision 为 None 时取 main 当前的 commit 并固定下来。"""
    from huggingface_hub import HfApi, snapshot_download
    info = HfApi().model_info(MODEL, revision=revision or "main", files_metadata=True)
    files = [s for s in info.siblings if wanted(s.rfilename)]
    if ON_PAI:
        try:
            from modelscope import snapshot_download as ms_download
            try:
                path = ms_download(MODEL, cache_dir=os.path.join(WORK, "ms_cache"), allow_patterns=WEIGHT_PATTERNS)
            except TypeError:  # 旧版 modelscope 的参数名
                path = ms_download(MODEL, cache_dir=os.path.join(WORK, "ms_cache"), allow_file_pattern=WEIGHT_PATTERNS)
            bad = []
            for s in files:
                f = os.path.join(path, s.rfilename)
                if not os.path.isfile(f):
                    bad.append(f"{s.rfilename}：ModelScope 上没有")
                    continue
                lfs = s.lfs
                want = (lfs["sha256"] if isinstance(lfs, dict) else lfs.sha256) if lfs else s.blob_id
                got = file_digest(f, "sha256") if lfs else file_digest(f, "sha1", b"blob %d\0" % os.path.getsize(f))
                if got != want:
                    bad.append(f"{s.rfilename}：内容不同")
            if bad or not files:
                raise RuntimeError(f"和 Hugging Face 快照 {info.sha} 对不上：{bad[:5]}")
            print(f"从 ModelScope 下载，{len(files)} 个文件和 Hugging Face 快照 {info.sha} 逐个核对一致", flush=True)
            return path, info.sha, "modelscope（已按 Hugging Face 快照核对）"
        except Exception as e:
            print(f"从 ModelScope 取模型没成功（{type(e).__name__}: {str(e)[:200]}），改从 {os.environ.get('HF_ENDPOINT', 'Hugging Face')} 下载", flush=True)
    path = snapshot_download(MODEL, revision=info.sha, allow_patterns=WEIGHT_PATTERNS)
    return path, info.sha, os.environ.get("HF_ENDPOINT", "https://huggingface.co")


# ---------- 概率、指标、评分 ----------

def softmax(row):
    if not all(math.isfinite(x) for x in row):
        raise ValueError(f"logits 里有非有限值：{row}")
    m = max(row)
    e = [math.exp(x - m) for x in row]
    s = sum(e)
    return [x / s for x in e]


def probs(lm, a, b, mode):
    pa = softmax(lm[(a, b)])
    if mode == "plain":
        return pa
    pb = softmax(lm[(b, a)])
    return [(pa[i] + pb[SWAP[i]]) / 2 for i in range(4)]


def argmax_label(p):
    if not all(math.isfinite(x) for x in p):
        raise ValueError(f"概率里有非有限值：{p}")  # 不让 NaN 经过 argmax 变成一个正常的标签
    return LABELS[max(range(4), key=lambda i: p[i])]


def weighted_f1(gold, pred):
    total, out = len(gold), 0.0
    for l in LABELS:
        tp = sum(g == l and p == l for g, p in zip(gold, pred))
        fp = sum(g != l and p == l for g, p in zip(gold, pred))
        fn = sum(g == l and p != l for g, p in zip(gold, pred))
        out += (2 * tp / (2 * tp + fp + fn) if tp else 0.0) * sum(g == l for g in gold) / total
    return out


def official_score(rows, labels, d, name):
    """官方脚本评分。金标子集只写四列（短语以撇号开头时 csv.Sniffer 会猜错格式）。"""
    part = name.rsplit("_", 1)[0]  # test_plain 和 test_sym 共用一份金标
    ref, pred, out = (os.path.abspath(os.path.join(d, x)) for x in (f"reference_{part}.csv", f"pred_{name}.csv", f"scores_{name}"))
    if not os.path.exists(ref):
        write_rows(ref, rows, REFERENCE_FIELDS)
    write_rows(pred, [{"instance_id": r["instance_id"], "label": l} for r, l in zip(rows, labels)], ["instance_id", "label"])
    res = subprocess.run([sys.executable, "-m", "evaluation_functions", "--gold", ref, "--predictions", pred, "--output-dir", out],
                         cwd=DICO_DIR, capture_output=True, text=True)
    if res.returncode != 0:
        raise SystemExit(f"官方评分脚本出错（{name}）：{res.stderr.strip()[-500:]}")
    s = read_json(os.path.join(out, "scores.json"))
    local = weighted_f1([r["label"] for r in rows], labels)
    assert abs(local - s["weighted_f1"]) < 1e-9, (name, local, s["weighted_f1"])
    return {k: s[k] for k in METRICS}


def save_logits(path, lm):
    write_json(path, {"inputs": [list(k) for k in lm], "logits": list(lm.values())})


def load_logits(path):
    d = read_json(path)
    return {tuple(k): v for k, v in zip(d["inputs"], d["logits"])}


# ---------- 损失（唯一和方法有关的部分；都在关掉 autocast 的范围里用 fp32 算） ----------

def fp32_scope():
    """关掉 autocast：在混合精度前向之后，用 fp32 算归一化、相似度矩阵和各项损失。"""
    import torch
    return torch.autocast(device_type="cuda", enabled=False)


def sym_kl(lp, lq):
    """(KL(p‖q) + KL(q‖p)) / 2，按行平均；输入是 log 概率。"""
    with fp32_scope():
        lp, lq = lp.float(), lq.float()
        p, q = lp.exp(), lq.exp()
        return 0.5 * ((p * (lp - lq)).sum(-1) + (q * (lq - lp)).sum(-1)).mean()


def supcon(q, q_labels, q_ids, keys, k_labels, k_ids, tau):
    """监督对比损失。锚点 q 的候选是 keys 里编号（训练记录的行号）不同的键，正例是其中标签相同的；
    L_i = -mean_{p∈P(i)} log( exp(q_i·k_p/τ) / Σ_{a∈A(i)} exp(q_i·k_a/τ) )，没有正例的锚点跳过。返回 (损失, 有效锚点数)。"""
    import torch
    with fp32_scope():
        q, keys = q.float(), keys.float()
        logits = q @ keys.T / tau
        valid = k_ids[None, :] != q_ids[:, None]
        pos = valid & (k_labels[None, :] == q_labels[:, None])
        logits = logits.masked_fill(~valid, float("-inf"))
        log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)
        npos = pos.sum(1)
        has = npos > 0
        if not bool(has.any()):
            return q.sum() * 0.0, 0
        per = -(log_prob.masked_fill(~pos, 0.0).sum(1)) / npos.clamp_min(1)
        return per[has].mean(), int(has.sum())


def kd_loss(logits, y, teacher_p, alpha, T):
    import torch
    import torch.nn.functional as F
    with fp32_scope():
        logits = logits.float()
        ce = F.cross_entropy(logits, y)
        t = torch.softmax(torch.log(teacher_p.float().clamp_min(1e-12)) / T, dim=-1)
        s_log = torch.log_softmax(logits / T, dim=-1)
        kl = (t * (torch.log(t.clamp_min(1e-12)) - s_log)).sum(-1).mean()
        return (1 - alpha) * ce + alpha * T * T * kl, {"ce": float(ce), "kl": float(kl)}


def apply_update(loss, opt, sched, scaler, params, use_scaler):
    """一次优化步。返回 (状态, 梯度范数)：
    updated：参数更新了，学习率日程前进一步；
    skipped：fp16 下梯度里有非有限的元素，梯度缩放跳过这次 optimizer.step()（正常现象），学习率日程不前进；
    nonfinite_grad：bf16 / fp32 下梯度里有非有限的元素，不调用 optimizer.step()，由调用方整次换精度重跑或报错；
    norm_overflow：梯度元素都有限、但总范数溢出成 Inf（梯度缩放发现不了这种情况），同样不调用 optimizer.step()，交给调用方处理。
    先单独检查梯度元素是否有限，再算范数，不用范数去推断梯度缩放是否跳过了更新。"""
    import torch
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    grads = [p.grad for p in params if p.grad is not None]
    elems_finite = bool(torch.stack([torch.isfinite(g).all() for g in grads]).all()) if grads else True
    if not elems_finite:
        if not use_scaler:
            opt.zero_grad(set_to_none=True)
            return "nonfinite_grad", float("nan")
        scaler.step(opt)  # unscale_ 时已经记下了非有限梯度，这里会跳过 optimizer.step()
        scaler.update()
        return "skipped", float("nan")
    gnorm = torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
    if not bool(torch.isfinite(gnorm)):
        opt.zero_grad(set_to_none=True)
        return "norm_overflow", float(gnorm)
    scaler.step(opt)
    scaler.update()
    sched.step()
    return "updated", float(gnorm)


def self_check_losses():
    """SMOKE 用：在真实的 torch 上，把各个损失和逐条循环写的参考实现比一遍；在 autocast 里检查损失仍是 fp32；
    检查非有限梯度时的更新逻辑和队列的环形写入。"""
    import torch
    g = torch.Generator().manual_seed(0)
    n, d = 7, 5
    q = torch.nn.functional.normalize(torch.randn(n, d, generator=g), dim=1)
    keys = torch.nn.functional.normalize(torch.randn(11, d, generator=g), dim=1)
    ql, qi = torch.tensor([0, 1, 1, 2, 3, 0, 2]), torch.arange(n)
    kl_, ki = torch.tensor([0, 1, 2, 3, 0, 1, 1, 2, 3, 0, 3]), torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
    got, _ = supcon(q, ql, qi, keys, kl_, ki, 0.3)
    tot, cnt = 0.0, 0
    for i in range(n):
        cand = [j for j in range(len(keys)) if int(ki[j]) != int(qi[i])]
        pos = [j for j in cand if int(kl_[j]) == int(ql[i])]
        if not pos:
            continue
        den = sum(math.exp(float(q[i] @ keys[j]) / 0.3) for j in cand)
        tot += -sum(math.log(math.exp(float(q[i] @ keys[j]) / 0.3) / den) for j in pos) / len(pos)
        cnt += 1
    assert abs(float(got) - tot / cnt) < 1e-4, ("supcon", float(got), tot / cnt)
    a, b = torch.randn(4, 4, generator=g), torch.randn(4, 4, generator=g)
    la, lb = torch.log_softmax(a, -1), torch.log_softmax(b, -1)
    ref = sum(0.5 * sum(math.exp(la[i, c]) * (la[i, c] - lb[i, c]) + math.exp(lb[i, c]) * (lb[i, c] - la[i, c]) for c in range(4))
              for i in range(4)) / 4
    assert abs(float(sym_kl(la, lb)) - float(ref)) < 1e-5, "sym_kl"
    tp = torch.softmax(torch.randn(4, 4, generator=g), -1)
    y = torch.tensor([0, 1, 2, 3])
    got, _ = kd_loss(a, y, tp, 0.5, 2.0)
    ref = 0.0
    for i in range(4):
        ce = -float(torch.log_softmax(a[i], -1)[y[i]])
        t = [math.exp(math.log(float(tp[i, c])) / 2) for c in range(4)]
        z = sum(t)
        t = [x / z for x in t]
        s = torch.log_softmax(a[i] / 2, -1)
        ref += 0.5 * ce + 0.5 * 4 * sum(t[c] * (math.log(t[c]) - float(s[c])) for c in range(4))
    assert abs(float(got) - ref / 4) < 1e-4, ("kd", float(got), ref / 4)
    pab = torch.randn(3, 4, generator=g)
    assert float(sym_kl(torch.log_softmax(pab, -1), torch.log_softmax(pab[:, SWAP], -1)[:, SWAP])) < 1e-6, "F/B 对调"
    q2 = Queue(5, 3, "cpu")
    for step in range(4):
        q2.push(torch.ones(2, 3) * step, torch.tensor([step, step]), torch.tensor([10 * step, 10 * step + 1]), step)
    assert q2.filled == 5 and sorted(q2.ids[: q2.filled].tolist()) == [11, 20, 21, 30, 31], q2.ids.tolist()
    if torch.cuda.is_available():
        dt = amp_dtype()
        qc, kc = q.cuda(), keys.cuda()
        with torch.autocast(device_type="cuda", dtype=dt):
            c1, _ = supcon(qc, ql.cuda(), qi.cuda(), kc, kl_.cuda(), ki.cuda(), 0.3)
            c2 = sym_kl(la.cuda(), lb.cuda())
            c3, _ = kd_loss(a.cuda(), y.cuda(), tp.cuda(), 0.5, 2.0)
        assert c1.dtype == c2.dtype == c3.dtype == torch.float32, (c1.dtype, c2.dtype, c3.dtype)
        assert abs(float(c1) - tot / cnt) < 1e-4, "autocast 里的 supcon 和 fp32 参考不同"
        # fp16 梯度缩放遇到非有限梯度：参数不变、学习率日程不前进
        w = torch.nn.Parameter(torch.ones(3, device="cuda"))
        opt = torch.optim.SGD([w], lr=0.1)
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda k: 1.0 / (k + 1))
        scaler = torch.amp.GradScaler("cuda")
        h = w.register_hook(lambda grad: grad * float("inf"))
        status, _ = apply_update((w * 2).sum(), opt, sched, scaler, [w], True)
        h.remove()
        assert status == "skipped" and torch.equal(w.detach(), torch.ones(3, device="cuda")) and sched.last_epoch == 0, status
        status, _ = apply_update((w * 2).sum(), opt, sched, scaler, [w], True)
        assert status == "updated" and sched.last_epoch == 1
        # bf16 / fp32（不用梯度缩放）遇到非有限梯度：不更新，返回 nonfinite_grad
        w2 = torch.nn.Parameter(torch.ones(3, device="cuda"))
        opt2 = torch.optim.SGD([w2], lr=0.1)
        sched2 = torch.optim.lr_scheduler.LambdaLR(opt2, lambda k: 1.0)
        h = w2.register_hook(lambda grad: grad * float("nan"))
        status, _ = apply_update((w2 * 2).sum(), opt2, sched2, torch.amp.GradScaler("cuda", enabled=False), [w2], False)
        h.remove()
        assert status == "nonfinite_grad" and torch.equal(w2.detach(), torch.ones(3, device="cuda")) and sched2.last_epoch == 0
        # 梯度元素都有限、总范数溢出（GPT 第二轮复现的边界）：fp16 和 bf16 下都不调用 optimizer.step()
        for scaled in (True, False):
            w3 = torch.nn.Parameter(torch.ones(4, device="cuda"))
            opt3 = torch.optim.AdamW([w3], lr=0.1, weight_decay=0.1)
            sched3 = torch.optim.lr_scheduler.LambdaLR(opt3, lambda k: 1.0)
            sc3 = torch.amp.GradScaler("cuda", enabled=scaled)
            h = w3.register_hook(lambda grad: torch.full_like(grad, 1e20 * (sc3.get_scale() if scaled else 1.0)))
            status, gn = apply_update(w3.sum(), opt3, sched3, sc3, [w3], scaled)
            h.remove()
            assert status == "norm_overflow" and math.isinf(gn), (scaled, status, gn)
            assert torch.equal(w3.detach(), torch.ones(4, device="cuda")) and not opt3.state and sched3.last_epoch == 0, scaled
    print("损失自检通过：监督对比、对称 KL、蒸馏、F/B 对调和逐条参考实现一致；autocast 里损失仍是 fp32；"
          "非有限梯度、梯度范数溢出时都不更新参数、学习率日程不前进；队列的环形写入正确", flush=True)


class Queue:
    """MoCo 的键队列：存键、标签、训练记录的行号和入队时的更新步数，环形覆盖最旧的。"""

    def __init__(self, size, dim, device):
        import torch
        self.size, self.ptr, self.filled = size, 0, 0
        self.keys = torch.zeros(size, dim, device=device)
        self.labels = torch.zeros(size, dtype=torch.long, device=device)
        self.ids = torch.full((size,), -1, dtype=torch.long, device=device)
        self.steps = torch.zeros(size, dtype=torch.long, device=device)

    def push(self, k, labels, ids, step):
        n = k.shape[0]
        idx = [(self.ptr + i) % self.size for i in range(n)]
        self.keys[idx] = k.detach().float()
        self.labels[idx] = labels
        self.ids[idx] = ids
        self.steps[idx] = step
        self.ptr = (self.ptr + n) % self.size
        self.filled = min(self.size, self.filled + n)

    def view(self):
        n = self.filled
        return self.keys[:n], self.labels[:n], self.ids[:n]

    def mean_age(self, step):
        return float((step - self.steps[: self.filled]).float().mean()) if self.filled else 0.0


# ---------- 训练 ----------

class SimulatedInterrupt(Exception):
    pass


def free_memory():
    import gc
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass


_MODEL_PATH = {}
_TOK = {}


def tokenizer():
    if "tok" not in _TOK:
        from transformers import AutoTokenizer
        _TOK["tok"] = AutoTokenizer.from_pretrained(_MODEL_PATH["path"])
    return _TOK["tok"]


def amp_dtype():
    import torch
    if not torch.cuda.is_available():
        return None
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def all_finite(lm):
    return all(math.isfinite(x) for row in lm.values() for x in row)


def train_run(t, parts, teacher=None, interrupt_after_epoch=None, max_steps=None):
    """训练一次，按内层验证选轮，返回 val / test / dev 的 logits 和训练记录。
    max_steps 只给 SMOKE 测速用：用正式的 12 轮学习率日程跑 max_steps 步（MoCo 先把队列填满），再测载入、推理和拷贝最佳状态的用时。"""
    import copy
    import numpy as np
    import torch
    import torch.nn.functional as F
    from torch.utils.data import DataLoader
    from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

    method, seed = t["method"], t["seed"]
    mp = method_params(method)
    epochs = MAX_EPOCHS if (max_steps or not SMOKE) else SMOKE_EPOCHS
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = tokenizer()
    fit, val = parts["fit"], parts["val"]
    if teacher is not None:
        assert len(teacher) == len(fit)

    def enc_of(t1, t2):
        e = tok(t1, t2, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
        return {k: v.to(device) for k, v in e.items()}

    @torch.no_grad()
    def logits_for(model, pairs):
        """相同输入只算一次；按长度排序成批；fp32 推理（不开 autocast）。"""
        model.eval()
        lens = [len(x) for x in tok([a for a, _ in pairs], [b for _, b in pairs], truncation=True, max_length=MAX_LEN)["input_ids"]]
        order = sorted(range(len(pairs)), key=lambda i: (lens[i], i))
        out = [None] * len(pairs)
        for s in range(0, len(order), EVAL_BATCH):
            idx = order[s: s + EVAL_BATCH]
            e = enc_of([pairs[i][0] for i in idx], [pairs[i][1] for i in idx])
            for i, row in zip(idx, model(**e).logits.float().cpu().tolist()):
                out[i] = row
        return dict(zip(pairs, out))

    val_inputs = needed_inputs(val, both=False)

    def run(dtype):
        """返回结果；出错时返回 {"failed": 原因, ...}，参数不会被非有限的梯度更新。"""
        t_start = time.time()
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        model = AutoModelForSequenceClassification.from_pretrained(_MODEL_PATH["path"], num_labels=4).float().to(device)
        hidden = model.config.hidden_size
        proj = target = target_proj = queue = None
        if method == "scl" or method.startswith("moco_m"):  # 在分类头之后建，分类头的初始化逻辑和其他方法相同
            proj = torch.nn.Sequential(torch.nn.Linear(hidden, PROJ["hidden"]), torch.nn.ReLU(),
                                       torch.nn.Linear(PROJ["hidden"], PROJ["out"])).to(device)
        if method.startswith("moco_m"):
            target, target_proj = copy.deepcopy(model).eval(), copy.deepcopy(proj).eval()
            for p in list(target.parameters()) + list(target_proj.parameters()):
                p.requires_grad_(False)
            queue = Queue(mp["queue"], PROJ["out"], device)
            if max_steps:  # 测速时先把队列填满，测的是满队列时的每步用时（这些随机键不进正式训练）
                gq = torch.Generator(device="cpu").manual_seed(0)
                queue.push(F.normalize(torch.randn(mp["queue"], PROJ["out"], generator=gq), dim=1).to(device),
                           torch.randint(0, 4, (mp["queue"],), generator=gq).to(device),
                           -torch.arange(2, mp["queue"] + 2).to(device), 0)
        named = list(model.named_parameters()) + ([(f"proj.{n}", p) for n, p in proj.named_parameters()] if proj is not None else [])
        no_decay = ("bias", "LayerNorm.weight", "layernorm.weight")
        groups = [{"params": [p for n, p in named if not any(k in n for k in no_decay)], "weight_decay": WEIGHT_DECAY},
                  {"params": [p for n, p in named if any(k in n for k in no_decay)], "weight_decay": 0.0}]
        params = [p for g_ in groups for p in g_["params"]]
        opt = torch.optim.AdamW(groups, lr=LR)
        loader = DataLoader(list(range(len(fit))), batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda b: b,
                            generator=torch.Generator().manual_seed(seed))
        total = epochs * len(loader)
        sched = get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO * total), total)
        use_scaler = dtype == torch.float16
        scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
        dir_set = {L2I[l] for l in DIR_RDROP["labels"]}
        best, best_epoch, best_state, history, updates, skipped_total = -1.0, 0, None, [], 0, 0
        if device == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        load_seconds = time.time() - t_start
        step_times = []

        def amp():
            return torch.autocast(device_type="cuda", dtype=dtype or torch.float32, enabled=dtype is not None)

        for epoch in range(epochs):
            model.train()
            if proj is not None:
                proj.train()
            t0 = time.time()
            stats = {"loss": [], "extra": [], "gnorm": [], "skipped": 0}
            for idx in loader:
                rows = [fit[i] for i in idx]
                y = torch.tensor([L2I[r["label"]] for r in rows], device=device)
                t1, t2 = [r["text1"] for r in rows], [r["text2"] for r in rows]
                ts = time.time()
                k = ids = None
                if method in ("ce", "kd"):
                    with amp():
                        lg = model(**enc_of(t1, t2)).logits
                    if method == "ce":
                        with fp32_scope():
                            loss, extra = F.cross_entropy(lg.float(), y), {}
                    else:
                        tp = torch.tensor([teacher[i] for i in idx], device=device, dtype=torch.float32)
                        loss, extra = kd_loss(lg, y, tp, mp["alpha"], mp["T"])
                elif method == "rdrop":
                    with amp():
                        lg = model(**enc_of(t1 + t1, t2 + t2)).logits  # 同一批拼成两份，两份的 dropout 掩码不同
                    with fp32_scope():
                        lg = lg.float()
                        l1, l2 = lg[: len(rows)], lg[len(rows):]
                        ce = (F.cross_entropy(l1, y) + F.cross_entropy(l2, y)) / 2
                        kl = sym_kl(torch.log_softmax(l1, -1), torch.log_softmax(l2, -1))
                        loss, extra = ce + mp["alpha"] * kl, {"ce": float(ce), "kl": float(kl)}
                elif method == "dir_rdrop":
                    sel = [i for i, r in enumerate(rows) if L2I[r["label"]] in dir_set]
                    with amp():
                        lg = model(**enc_of(t1 + [t2[i] for i in sel], t2 + [t1[i] for i in sel])).logits
                    with fp32_scope():
                        lg = lg.float()
                        lab = lg[: len(rows)]
                        ce = F.cross_entropy(lab, y)
                        if sel:
                            kl = sym_kl(torch.log_softmax(lab[sel], -1), torch.log_softmax(lg[len(rows):], -1)[:, SWAP])
                        else:
                            kl = lab.sum() * 0.0
                        loss, extra = ce + mp["alpha"] * kl, {"ce": float(ce), "kl": float(kl), "sel_frac": len(sel) / len(rows)}
                else:  # scl / moco：分类和对比用同一次前向；编码器和投影在 autocast 里，归一化和损失在外面用 fp32
                    e = enc_of(t1, t2)
                    with amp():
                        out = model(**e, output_hidden_states=True)
                        z = proj(out.hidden_states[-1][:, 0])
                        if queue is not None:
                            with torch.no_grad():
                                kz = target_proj(target(**e, output_hidden_states=True).hidden_states[-1][:, 0])
                    ids = torch.tensor(idx, device=device)
                    with fp32_scope():
                        ce = F.cross_entropy(out.logits.float(), y)
                        q = F.normalize(z.float(), dim=1)
                        if queue is None:
                            con, n_anchor = supcon(q, y, ids, q, y, ids, mp["tau"])
                        else:
                            k = F.normalize(kz.float(), dim=1)
                            qk, ql, qi = queue.view()
                            con, n_anchor = supcon(q, y, ids, torch.cat([k, qk]), torch.cat([y, ql]), torch.cat([ids, qi]), mp["tau"])
                        loss = (1 - mp["lambda"]) * ce + mp["lambda"] * con
                        extra = {"ce": float(ce), "con": float(con), "anchors": n_anchor / len(rows)}
                if not torch.isfinite(loss):
                    return {"failed": "nonfinite_loss", "epoch": epoch + 1, "updates": updates}
                status, gnorm = apply_update(loss, opt, sched, scaler, params, use_scaler)
                if status in ("nonfinite_grad", "norm_overflow"):
                    return {"failed": status, "epoch": epoch + 1, "updates": updates}
                if status == "skipped":  # fp16 跳过的更新：学习率日程、动量更新、入队都不前进
                    stats["skipped"] += 1
                else:
                    updates += 1
                    stats["gnorm"].append(gnorm)
                    if queue is not None:
                        with torch.no_grad():
                            m = mp["m"]
                            for pt, po in zip(list(target.parameters()) + list(target_proj.parameters()),
                                              list(model.parameters()) + list(proj.parameters())):
                                pt.mul_(m).add_(po.detach(), alpha=1 - m)
                        queue.push(k, y, ids, updates)
                if device == "cuda":
                    torch.cuda.synchronize()
                step_times.append(time.time() - ts)
                stats["loss"].append(float(loss))
                stats["extra"].append(extra)
                if max_steps and len(step_times) >= max_steps:
                    sample = (val_inputs + needed_inputs(parts["test"]))[:512]
                    t_inf = time.time()
                    logits_for(model, sample)
                    infer = (time.time() - t_inf) / max(1, len(sample))
                    t_clone = time.time()
                    _ = {k_: v.detach().cpu().clone() for k_, v in model.state_dict().items()}
                    return {"step_times": step_times, "load_seconds": load_seconds, "infer_seconds_per_input": infer,
                            "clone_seconds": time.time() - t_clone}
            lm = logits_for(model, val_inputs)
            if not all_finite(lm):
                return {"failed": "nonfinite_val_logits", "epoch": epoch + 1, "updates": updates}
            f1 = weighted_f1([r["label"] for r in val], [argmax_label(softmax(lm[(r["text1"], r["text2"])])) for r in val])
            rec = {"epoch": epoch + 1, "train_loss": sum(stats["loss"]) / len(stats["loss"]), "val_weighted_f1": f1,
                   "updates": len(stats["gnorm"]), "skipped_updates": stats["skipped"],
                   "grad_norm_mean": sum(stats["gnorm"]) / len(stats["gnorm"]) if stats["gnorm"] else None,
                   "grad_norm_max": max(stats["gnorm"]) if stats["gnorm"] else None, "seconds": round(time.time() - t0, 1)}
            for key in sorted({k_ for e_ in stats["extra"] for k_ in e_}):
                vals = [e_[key] for e_ in stats["extra"] if key in e_]
                rec[key] = sum(vals) / len(vals)
            if queue is not None:
                rec["queue_filled"] = queue.filled
                rec["queue_mean_key_age_updates"] = queue.mean_age(updates)
            skipped_total += stats["skipped"]
            history.append(rec)
            print(f"    第 {epoch + 1} 轮：loss {rec['train_loss']:.4f}  内层验证 WF1 {f1:.4f}  梯度范数均值 {rec['grad_norm_mean'] or float('nan'):.2f}"
                  + (f"  跳过的更新 {stats['skipped']}" if stats["skipped"] else "") + f"  {rec['seconds']} 秒", flush=True)
            if f1 > best:
                best, best_epoch = f1, epoch + 1
                best_state = {k_: v.detach().cpu().clone() for k_, v in model.state_dict().items()}
            if interrupt_after_epoch == epoch + 1:
                raise SimulatedInterrupt()
        model.load_state_dict(best_state)
        result = {"history": history, "best_epoch": best_epoch, "val_weighted_f1": best, "updates": updates,
                  "skipped_updates": skipped_total, "seconds_per_step": round(sum(step_times) / max(1, len(step_times)), 4),
                  "steps": len(step_times), "steps_per_epoch": len(loader)}
        result["val_logits"] = logits_for(model, val_inputs)
        result["test_logits"] = logits_for(model, needed_inputs(parts["test"]))
        if "dev" in parts:
            result["dev_logits"] = logits_for(model, needed_inputs(parts["dev"]))
        for name in ("val_logits", "test_logits", "dev_logits"):
            if name in result and not all_finite(result[name]):
                return {"failed": f"nonfinite_{name}", "epoch": best_epoch, "updates": updates}
        if device == "cuda":  # 峰值显存在最终推理之后读，覆盖整个任务
            result["peak_memory_gib"] = round(torch.cuda.max_memory_allocated() / 2 ** 30, 2)
            result["peak_reserved_gib"] = round(torch.cuda.max_memory_reserved() / 2 ** 30, 2)
        else:
            result["peak_memory_gib"] = result["peak_reserved_gib"] = None
        del model, best_state, target, opt
        return result

    names = {None: "fp32", torch.bfloat16: "bf16 autocast", torch.float16: "fp16 autocast"}
    dtype = amp_dtype()
    attempts = []
    for dt in ([dtype, None] if dtype is not None else [None]):
        res = run(dt)
        free_memory()
        if max_steps or "failed" not in res:
            if not max_steps:
                res["precision"] = names[dt] if not attempts else f"{names[dt]}（{attempts[0]['precision']} 失败后重跑：{attempts[0]['failed']}）"
                res["precision_attempts"] = attempts + [{"precision": names[dt], "result": "ok"}]
            return res
        attempts.append({"precision": names[dt], **res})
        print(f"    {names[dt]} 失败（{res['failed']}，第 {res['epoch']} 轮，已更新 {res['updates']} 次），"
              + ("整次改用 fp32 重跑" if dt is not None else "fp32 也失败"), flush=True)
    raise RuntimeError(f"{task_id(t)}：fp32 重跑也失败了：{attempts}")


TRAIN_FN = train_run  # 测试时可以换成假的训练函数


ENV = {}  # 这次运行的环境身份（卡、精度、torch 等），main 里填好，写进每个 done.json


def log_attempt(event, t, attempt_id, **info):
    """attempts.jsonl：每个任务每次尝试的开始、完成、失败、被打断都记一行，随结果包一起带走。"""
    os.makedirs(OUT, exist_ok=True)
    with open(os.path.join(OUT, "attempts.jsonl"), "a", encoding="utf-8") as f:
        f.write(json.dumps({"event": event, "task": task_id(t), "attempt_id": attempt_id, "session": RUN_STAMP,
                            "time": time.strftime("%Y-%m-%d %H:%M:%S"), **info}, ensure_ascii=False) + "\n")


def previous_attempts(t):
    path = os.path.join(OUT, "attempts.jsonl")
    if not os.path.exists(path):
        return 0
    with open(path, encoding="utf-8") as f:
        return sum(1 for line in f if line.strip() and json.loads(line)["task"] == task_id(t) and json.loads(line)["event"] == "start")


def run_task(t, revision, code_hash, interrupt_after_epoch=None, d=None):
    """d 只给 SMOKE 的中断自检用：放在单独的目录里，每次都重建，不碰正式任务的目录。"""
    d = d or task_dir(t)
    parts = parts_for(t["track"], t["fold"], t["with_dev"])
    teacher, extra = None, {}
    if t["method"] == "kd":
        teacher, tsha = load_teacher(t["seed"], parts["fit"])
        extra["teacher_sha256"] = tsha
    fp = task_fingerprint(t, revision, extra)
    if os.path.exists(os.path.join(d, "done.json")):
        old = read_json(os.path.join(d, "task.json"))
        if old != fp:
            diff = sorted(k for k in set(fp) | set(old) if fp.get(k) != old.get(k))
            raise SystemExit(f"{task_id(t)} 已完成，但设置和现在不同：{diff}。不把两种设置的结果混在一起，请检查。")
        return "skip"
    n_before = previous_attempts(t)
    attempt_id = f"{RUN_STAMP}-{task_id(t).replace('/', '-')}-{n_before + 1}"
    if os.path.isdir(d):
        print(f"  {task_id(t)} 上次没跑完，从头重跑", flush=True)
        log_attempt("incomplete_found", t, attempt_id, note="目录在、没有 done.json，删掉后从头重跑")
        shutil.rmtree(d)
    write_json(os.path.join(d, "task.json"), fp)
    log_attempt("start", t, attempt_id, code_sha256=code_hash, env=ENV)
    t0 = time.time()
    print(f"开始：{task_id(t)}（训练 {len(parts['fit'])} 行，内层验证 {len(parts['val'])} 行，"
          f"被预测的折 {len(parts['test'])} 行{'，dev ' + str(len(parts['dev'])) + ' 行' if 'dev' in parts else ''}）", flush=True)
    try:
        res = TRAIN_FN(t, parts, teacher=teacher, interrupt_after_epoch=interrupt_after_epoch)
    except SimulatedInterrupt:
        log_attempt("interrupted", t, attempt_id, note="SMOKE 的模拟中断")
        raise
    except BaseException as e:
        log_attempt("failed", t, attempt_id, error=f"{type(e).__name__}: {str(e)[:300]}")
        raise
    save_logits(os.path.join(d, "logits_val.json"), res["val_logits"])
    save_logits(os.path.join(d, "logits_test.json"), res["test_logits"])
    scores = {}
    for name in ("test", "dev"):
        if f"{name}_logits" not in res:
            continue
        if name == "dev":
            save_logits(os.path.join(d, "logits_dev.json"), res["dev_logits"])
        rows = parts[name]
        for mode in ("plain", "sym"):
            labels = [argmax_label(probs(res[f"{name}_logits"], r["text1"], r["text2"], mode)) for r in rows]
            scores.setdefault(name, {})[mode] = official_score(rows, labels, d, f"{name}_{mode}")
    write_json(os.path.join(d, "history.json"), res["history"])
    done = {"task": task_id(t), "attempt_id": attempt_id, "previous_attempts": n_before, "session": RUN_STAMP, "env": ENV,
            "code_sha256": code_hash, "best_epoch": res["best_epoch"], "val_weighted_f1": res["val_weighted_f1"], "scores": scores,
            "precision": res["precision"], "precision_attempts": res["precision_attempts"],
            "updates": res["updates"], "skipped_updates": res["skipped_updates"],
            "peak_memory_gib": res["peak_memory_gib"], "peak_reserved_gib": res["peak_reserved_gib"],
            "seconds_per_step": res["seconds_per_step"], "steps": res["steps"], "steps_per_epoch": res["steps_per_epoch"],
            "rows": {k: len(v) for k, v in parts.items()}, "minutes": round((time.time() - t0) / 60, 2)}
    write_json(os.path.join(d, "done.json"), done)
    log_attempt("completed", t, attempt_id, minutes=done["minutes"], precision=res["precision"])
    s = scores["test"]["sym"]
    print(f"完成：{task_id(t)}  选中第 {res['best_epoch']} 轮  被预测的折 sym {s['weighted_f1']:.3f} / {s['soft_cons']:.3f} / {s['hard_cons']:.3f}"
          + (f"  dev sym {scores['dev']['sym']['weighted_f1']:.3f}" if "dev" in scores else "")
          + f"  {res['precision']}  用时 {done['minutes']} 分钟", flush=True)
    return "done"


# ---------- 汇总 ----------

def assemble_oof(track, method, seed):
    """5 折都完成时，每行取它所在折的模型的 logits，拼成整个训练集的折外预测，用官方脚本算一次。"""
    dirs = [os.path.join(OUT, f"track{track}/{method}/seed{seed}/fold{k}") for k in range(FOLDS)]
    if not all(os.path.exists(os.path.join(x, "done.json")) for x in dirs):
        return None
    out_dir = os.path.join(OUT, f"track{track}/{method}/seed{seed}/oof")
    dones = [read_json(os.path.join(x, "done.json")) for x in dirs]
    mixed = {k: sorted({str(x.get(k)) for x in dones}) for k in ("code_sha256", "precision")}
    mixed = {k: v for k, v in mixed.items() if len(v) > 1}
    if mixed:  # 不把不同代码或不同精度跑出来的折拼成一份折外预测
        return {"not_assembled": mixed}
    if os.path.exists(os.path.join(out_dir, "done.json")):
        return read_json(os.path.join(out_dir, "done.json"))["scores"]
    fam_of, fold_of = split_base()
    rows = read_rows(data_file(track, "train", "reference"))
    lms = [load_logits(os.path.join(x, "logits_test.json")) for x in dirs]
    scores = {}
    for mode in ("plain", "sym"):
        labels = [argmax_label(probs(lms[fold_of[fam_of[r["pair_id"]]]], r["text1"], r["text2"], mode)) for r in rows]
        scores[mode] = official_score(rows, labels, out_dir, f"oof_{mode}")
    write_json(os.path.join(out_dir, "done.json"), {"scores": scores, "rows": len(rows)})
    return scores


def summarize():
    summary = {"version": VERSION, "stage": STAGE, "runs": {}, "oof": {}}
    for p in sorted(glob.glob(os.path.join(OUT, "track*/*/seed*/fold*/done.json"))):
        d = read_json(p)
        summary["runs"][d["task"]] = {k: d.get(k) for k in ("best_epoch", "val_weighted_f1", "scores", "precision", "minutes",
                                                            "seconds_per_step", "skipped_updates", "peak_memory_gib", "previous_attempts")}
    for track_dir in sorted(glob.glob(os.path.join(OUT, "track*"))):
        track = int(os.path.basename(track_dir)[5:])
        for method_dir in sorted(glob.glob(os.path.join(track_dir, "*"))):
            for seed_dir in sorted(glob.glob(os.path.join(method_dir, "seed*"))):
                sc = assemble_oof(track, os.path.basename(method_dir), int(os.path.basename(seed_dir)[4:]))
                if sc and "not_assembled" in sc:
                    summary.setdefault("oof_not_assembled", {})[f"track{track}/{os.path.basename(method_dir)}/{os.path.basename(seed_dir)}"] = sc["not_assembled"]
                elif sc:
                    summary["oof"][f"track{track}/{os.path.basename(method_dir)}/{os.path.basename(seed_dir)}"] = sc
    write_json(os.path.join(OUT, "summary.json"), summary)
    groups = {}
    for tid, r in summary["runs"].items():
        track, method, seed, fold = tid.split("/")
        if fold == "fold0":
            groups.setdefault((track, method), []).append(r)
    print("\n========== fold 0 汇总（对称化 WF1 / SoftCons / HardCons，种子平均；dev 只报告） ==========")
    for (track, method), rs in sorted(groups.items()):
        m = {k: sum(r["scores"]["test"]["sym"][k] for r in rs) / len(rs) for k in METRICS}
        dev = [r["scores"]["dev"]["sym"]["weighted_f1"] for r in rs if "dev" in r["scores"]]
        print(f"{track} {method:12s} {len(rs)} 个种子  fold 0 {m['weighted_f1']:.3f} / {m['soft_cons']:.3f} / {m['hard_cons']:.3f}"
              + (f"  dev WF1 {sum(dev) / len(dev):.3f}" if dev else ""))
    fallback = [k for k, r in summary["runs"].items() if r["precision"] and "重跑" in r["precision"]]
    if fallback:
        print("注意：这些任务回退到了 fp32，和混合精度的基线不是同一环境：", fallback)
    if summary.get("oof_not_assembled"):
        print("注意：这些种子的 5 折代码或精度不一致，没有拼成折外预测：", summary["oof_not_assembled"])
    if summary["oof"]:
        print("5 折拼起来的折外预测（对称化 WF1）：")
        for k, sc in summary["oof"].items():
            print(f"  {k}: {sc['sym']['weighted_f1']:.4f}")
    return summary


# ---------- SMOKE：自检、中断续跑、测速、时间估计 ----------

def smoke_extras(revision, code_hash):
    report = {}
    self_check_losses()
    report["loss_self_check"] = "通过"
    for s in SEEDS_STAGE1:  # 老师概率要覆盖真实的 2,190 行，不只是 SMOKE 子集；身份逐项核对
        probs_, tsha = load_teacher(s, parts_for(1, 0, False, smoke=False)["fit"])
        report[f"teacher_seed{s}"] = {"rows": len(probs_), "sha256": tsha}
    print("老师概率：三个种子的身份核对通过，覆盖全部 2,190 行", flush=True)
    # 中断自检放在单独的目录里，每次 SMOKE 都重建，重复运行 SMOKE 也不受影响
    t = make_task(1, "ce", 43, 0, True)
    d = os.path.join(OUT, "selfcheck", "interrupt")
    shutil.rmtree(d, ignore_errors=True)
    try:
        run_task(t, revision, code_hash, interrupt_after_epoch=1, d=d)
        raise AssertionError("模拟中断没有发生")
    except SimulatedInterrupt:
        free_memory()
    assert os.path.isdir(d) and not os.path.exists(os.path.join(d, "done.json"))
    run_task(t, revision, code_hash, d=d)
    assert os.path.exists(os.path.join(d, "done.json"))
    report["interrupt_resume"] = "通过：第 1 轮后模拟中断，重跑时从头训练并完成（单独的自检目录）"
    print("中断续跑检查通过", flush=True)
    timing = {}
    probe = [(1, m) for m in STAGE1_TRACK1] + [(2, "ce"), (3, "ce"), (4, "ce")]
    for track, method in probe:
        full = parts_for(track, 0, True, smoke=False)  # 测速用正式规模的数据、batch 和 12 轮的学习率日程，只跑 TIMING_STEPS 步
        teacher = load_teacher(42, full["fit"])[0] if method == "kd" else None
        res = TRAIN_FN(make_task(track, method, 42, 0, False), full, teacher=teacher, max_steps=TIMING_STEPS)
        if "failed" in res:
            raise SystemExit(f"测速时 track{track} {method} 失败：{res}")
        per = sum(res["step_times"][3:]) / len(res["step_times"][3:])  # 前 3 步含初始化和缓存分配，不计
        timing[f"track{track}/{method}"] = {
            "seconds_per_step": round(per, 4), "steps": len(res["step_times"]) - 3, "load_seconds": round(res["load_seconds"], 1),
            "infer_seconds_per_input": res["infer_seconds_per_input"], "clone_seconds": round(res["clone_seconds"], 2),
            "inputs": {"val": len(needed_inputs(full["val"], both=False)), "test": len(needed_inputs(full["test"])),
                       "dev": len(needed_inputs(full["dev"]))}}
        print(f"  测速 track{track} {method:12s} 每步 {per:.3f} 秒（{len(res['step_times']) - 3} 步，不计前 3 步）"
              f"；载入 {res['load_seconds']:.0f} 秒；推理每千条 {1000 * res['infer_seconds_per_input']:.1f} 秒", flush=True)
        del res
        free_memory()
    report["timing"] = timing
    report["estimate"] = estimate(timing)
    write_json(os.path.join(OUT, "smoke_report.json"), report)
    free_memory()
    return report


def estimate(timing):
    """按 SMOKE 实测估计正式运行的时间（性能采样，不是完整配方的复现；第一阶段跑完后用完整任务的实际用时更新）。
    每次运行 = 训练步数 × 每步秒数 + 载入 + 每轮的内层验证推理和最佳状态拷贝 + 最终推理（val、被预测的折、fold 0 时加 dev）+ 评分约 10 秒。"""
    def run_min(track, method, with_dev=True):
        x = timing[f"track{track}/{method}"]
        n_fit = 2190 if track != 4 else 13140
        steps = math.ceil(n_fit / BATCH_SIZE) * MAX_EPOCHS
        n = x["inputs"]
        per_epoch = n["val"] * x["infer_seconds_per_input"] + x["clone_seconds"]
        final = (n["val"] + n["test"] + (n["dev"] if with_dev else 0)) * x["infer_seconds_per_input"]
        return (steps * x["seconds_per_step"] + x["load_seconds"] + MAX_EPOCHS * per_epoch + final + 10) / 60

    est = {"per_run_minutes": {k: round(run_min(int(k[5]), k.split("/")[1]), 2) for k in timing}}
    s1 = sum(run_min(1, m) for m in STAGE1_TRACK1) * len(SEEDS_STAGE1) + sum(run_min(t, "ce") for t in (2, 3, 4)) * len(SEEDS_STAGE1)
    est["stage1_minutes"] = round(s1, 1)
    worst_m = max(MOCO["m_candidates"], key=lambda m: timing[f"track1/moco_m{m}"]["seconds_per_step"])
    methods = ["ce", "rdrop", "dir_rdrop", "scl", f"moco_m{worst_m}"]
    n_new_fold0 = len(SEEDS_STAGE2) - len(SEEDS_STAGE1)
    n_new_other = len(SEEDS_STAGE2) * (FOLDS - 1)
    est["stage2_minutes"] = round(sum(run_min(1, m) for m in methods) * n_new_fold0 + sum(run_min(1, m, False) for m in methods) * n_new_other, 1)
    print(f"\n按实测估计：第一阶段约 {est['stage1_minutes']:.0f} 分钟，第二阶段约 {est['stage2_minutes']:.0f} 分钟"
          "（不含装环境、下载模型；第一阶段跑完后会用完整任务的实际用时更新）", flush=True)
    return est


# ---------- 主流程 ----------

def main():
    ensure_packages()
    ensure_official_repo()
    restore_previous_results()
    common = check_common_fingerprint()
    revision = common.get("model_revision")
    path, revision_now, source = fetch_model(revision)
    if revision is None:
        common["model_revision"] = revision_now
        write_json(os.path.join(OUT, "fingerprint.json"), common)
    revision = revision_now
    _MODEL_PATH["path"] = path
    import torch
    if not torch.cuda.is_available() and os.environ.get("DICO_ALLOW_CPU") != "1":
        raise SystemExit("没有可用的 GPU（torch.cuda.is_available() 是 False）。正式运行不在 CPU 上跑，先停下；请检查实例的卡和驱动。")
    dtype = amp_dtype()
    ENV.update({**env_info(), "platform": PLATFORM, "amp": {None: "fp32", torch.bfloat16: "bf16", torch.float16: "fp16"}[dtype]})
    code_hash = record_run({**ENV, "model_revision": revision, "model_source": source})
    print("模型快照：", revision, "；来源：", source, "；混合精度：", ENV["amp"], "；卡：", ENV["gpu"], flush=True)
    tasks = stage_tasks(STAGE)
    if any(t["method"] == "kd" for t in tasks):  # 开跑前先确认老师概率在，免得跑到一半才停
        for s in sorted({t["seed"] for t in tasks if t["method"] == "kd"}):
            try:
                load_teacher(s, parts_for(1, 0, False, smoke=False)["fit"])
            except FileNotFoundError as e:
                raise SystemExit(f"{e}。请把上传包里的 dico_qwen_teacher_results 放到 {TEACHER_DIR}（sh 的 setup 会自动放好）。")
    finished = False
    try:
        if SMOKE:
            smoke_extras(revision, code_hash)
        for i, t in enumerate(tasks, 1):
            status = run_task(t, revision, code_hash)
            if status == "done":
                pack(force=False)
            print(f"[{i}/{len(tasks)}] {task_id(t)}：{'已完成，跳过' if status == 'skip' else '完成'}", flush=True)
        summarize()
        finished = True
    finally:  # 出错时也把已经完成的任务打进结果包
        try:
            zip_path = pack()
            print("\n结果包：", zip_path, flush=True)
        except Exception as e:
            print("打结果包失败：", e, flush=True)
        if not finished:
            print("运行中途出错了（见上面的报错）。已完成的任务都在结果包里；请把结果包和日志发给 Claude。", flush=True)
    if SMOKE:
        print("SMOKE 跑完了：损失自检、中断续跑、各方法的小规模训练都通过。请把结果包和上面的输出发给 Claude，确认后再跑第一阶段。")
    elif STAGE == "stage1":
        print("第一阶段跑完了。请下载这个结果包发给 Claude；第二阶段在下一次开实例时跑（需要这个结果包）。")
    else:
        print("第二阶段跑完了。请下载这个结果包发给 Claude。")


if __name__ == "__main__":
    main()